# 03 - Salary Prediction Model

Crear Salaries.csv


In [4]:
import pandas as pd
from pathlib import Path

RAW       = Path('../data/raw')
PROCESSED = Path('../data/processed')

df = pd.read_csv(
    RAW / 'survey_results_public.csv',
    usecols=['Country', 'DevType', 'ConvertedCompYearly', 'WorkExp', 
             'EdLevel', 'Employment', 'LanguageHaveWorkedWith']
)

# Filtrar: solo empleados con salario válido
df = df[
    (df['Employment'] == 'Employed') &
    df['ConvertedCompYearly'].notna() &
    (df['ConvertedCompYearly'] > 5000) &
    (df['ConvertedCompYearly'] < 500000)
].copy()

# Limpiar WorkExp
df['WorkExp'] = pd.to_numeric(df['WorkExp'], errors='coerce')
df = df.dropna(subset=['WorkExp'])

# Primer rol mencionado en DevType
df['role'] = df['DevType'].str.split(';').str[0].str.strip()
df = df.dropna(subset=['role'])

# Simplificar EdLevel
def map_ed(val):
    if pd.isna(val): return 'Bachelor'
    val = str(val).lower()
    if 'master' in val: return 'Master'
    if 'doctor' in val or 'phd' in val: return 'PhD'
    if 'bachelor' in val: return 'Bachelor'
    return 'Bootcamp'

df['education_level'] = df['EdLevel'].apply(map_ed)

# Construir salaries.csv
USD_TO_EUR = 0.92
df_salaries = pd.DataFrame({
    'city_name':        df['Country'],
    'role':             df['role'],
    'specialization':   df['specialization'],
    'years_experience': df['WorkExp'].astype(int),
    'education_level':  df['education_level'],
    'gross_salary':     (df['ConvertedCompYearly'] * USD_TO_EUR).round(0).astype(int),
})

df_salaries = df_salaries.dropna()
df_salaries.to_csv(PROCESSED / 'salaries.csv', index=False)

print(f"Registros:       {len(df_salaries)}")
print(f"Países únicos:   {df_salaries['city_name'].nunique()}")
print(f"Roles únicos:    {df_salaries['role'].nunique()}")
df_salaries.head(5)

KeyError: 'specialization'

Entrenamiento Del ML


In [7]:
import sys
sys.path.insert(0, '..')

import pandas as pd
from pathlib import Path
from src.model import train

PROCESSED = Path('../data/processed')

df_salaries = pd.read_csv(PROCESSED / 'salaries.csv')
print(f"Registros: {len(df_salaries)}")
print(f"Columnas: {df_salaries.columns.tolist()}")
df_salaries.head(3)

Registros: 18112
Columnas: ['city_name', 'role', 'specialization', 'years_experience', 'education_level', 'gross_salary']


,city_name,role,specialization,years_experience,education_level,gross_salary
0,Ukraine,"Developer, mobile",General,8,Master,56356
1,Netherlands,"Developer, back-end",General,2,Bootcamp,96060
2,Ukraine,"Developer, back-end",General,4,Bachelor,33301


In [ ]:
metrics = train(df_salaries, target='gross_salary', model_type='gbr')
print(f"MAE:  €{metrics['mae']:,.0f}")
print(f"R²:   {metrics['r2']}")

Modelo guardado: c:\Users\Damián\Desktop\RelocationDevs\notebooks\..\models\salary_model.pkl
MAE:  €29,111
R²:   0.5306


Maperar lenguajes en especialización


In [ ]:
# Mapear lenguajes/tecnologías a especialización
def map_specialization(languages):
    if pd.isna(languages):
        return 'General'
    lang = str(languages).lower()
    if any(x in lang for x in ['python', 'r ', 'tensorflow', 'pytorch', 'spark']):
        return 'AI/ML'
    if any(x in lang for x in ['javascript', 'typescript', 'react', 'vue', 'angular', 'html', 'css']):
        return 'Frontend'
    if any(x in lang for x in ['java', 'kotlin', 'c#', 'go', 'rust', 'c++']):
        return 'Backend'
    if any(x in lang for x in ['aws', 'azure', 'gcp', 'docker', 'kubernetes', 'terraform']):
        return 'Cloud'
    if any(x in lang for x in ['swift', 'dart', 'flutter', 'react native']):
        return 'Mobile'
    if any(x in lang for x in ['sql', 'postgresql', 'mysql', 'mongodb', 'databricks']):
        return 'Data Engineer'
    return 'General'

df['specialization'] = df['LanguageHaveWorkedWith'].apply(map_specialization)

In [5]:
import pandas as pd
from pathlib import Path

RAW       = Path('../data/raw')
PROCESSED = Path('../data/processed')

df = pd.read_csv(
    RAW / 'survey_results_public.csv',
    usecols=['Country', 'DevType', 'ConvertedCompYearly', 'WorkExp', 
             'EdLevel', 'Employment', 'LanguageHaveWorkedWith']
)

In [6]:
df = df[
    (df['Employment'] == 'Employed') &
    df['ConvertedCompYearly'].notna() &
    (df['ConvertedCompYearly'] > 5000) &
    (df['ConvertedCompYearly'] < 500000)
].copy()

df['WorkExp'] = pd.to_numeric(df['WorkExp'], errors='coerce')
df = df.dropna(subset=['WorkExp'])
df['role'] = df['DevType'].str.split(';').str[0].str.strip()
df = df.dropna(subset=['role'])

In [7]:
def map_ed(val):
    if pd.isna(val): return 'Bachelor'
    val = str(val).lower()
    if 'master' in val: return 'Master'
    if 'doctor' in val or 'phd' in val: return 'PhD'
    if 'bachelor' in val: return 'Bachelor'
    return 'Bootcamp'

def map_specialization(languages):
    if pd.isna(languages): return 'General'
    lang = str(languages).lower()
    if any(x in lang for x in ['python', 'tensorflow', 'pytorch', 'spark']): return 'AI/ML'
    if any(x in lang for x in ['javascript', 'typescript', 'react', 'vue', 'html']): return 'Frontend'
    if any(x in lang for x in ['java', 'kotlin', 'c#', 'go', 'rust']): return 'Backend'
    if any(x in lang for x in ['aws', 'azure', 'gcp', 'docker', 'kubernetes']): return 'Cloud'
    if any(x in lang for x in ['swift', 'dart', 'flutter']): return 'Mobile'
    if any(x in lang for x in ['sql', 'postgresql', 'mysql', 'mongodb']): return 'Data Engineer'
    return 'General'

df['education_level']  = df['EdLevel'].apply(map_ed)
df['specialization']   = df['LanguageHaveWorkedWith'].apply(map_specialization)

In [8]:
USD_TO_EUR = 0.92
df_salaries = pd.DataFrame({
    'city_name':        df['Country'],
    'role':             df['role'],
    'specialization':   df['specialization'],
    'years_experience': df['WorkExp'].astype(int),
    'education_level':  df['education_level'],
    'gross_salary':     (df['ConvertedCompYearly'] * USD_TO_EUR).round(0).astype(int),
})
df_salaries = df_salaries.dropna()
df_salaries.to_csv(PROCESSED / 'salaries.csv', index=False)
print(f"Registros: {len(df_salaries)}")
print(f"Especializaciones: {df_salaries['specialization'].value_counts().to_dict()}")

Registros: 18112
Especializaciones: {'AI/ML': 9810, 'Frontend': 5658, 'General': 1390, 'Backend': 1039, 'Data Engineer': 134, 'Mobile': 81}


In [9]:
import sys
sys.path.insert(0, '..')
from src.model import train

metrics = train(df_salaries, target='gross_salary', model_type='gbr')
print(f"MAE:  €{metrics['mae']:,.0f}")
print(f"R²:   {metrics['r2']}")

Modelo guardado: c:\Users\Damián\Desktop\RelocationDevs\notebooks\..\models\salary_model.pkl
MAE:  €29,022
R²:   0.5347
